# RoadGuard — Model Training and Evaluation

Run the LOFO folds: for each held-out family, an RGB+depth teacher, an RGB baseline and a distilled RGB student. Training, selection and evaluation live in `roadguard`; everything that varies between runs lives in a YAML under `configs/`.

## 1. Mount Drive and install RoadGuard

Use a GPU Colab runtime. Clone the repository if needed, or pull the latest commit before proceeding. SegFormer needs the `transformers` version pinned by the repository; do not upgrade packages in this notebook.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os
import subprocess
import base64

# Mount Google Drive
drive.mount("/content/drive")

# Clone private RoadGuard repository
REPO = Path("/content/road-guard")

if not REPO.exists():
    token = userdata.get("GITHUB_TOKEN")

    credentials = base64.b64encode(
        f"x-access-token:{token}".encode()
    ).decode()

    env = os.environ.copy()
    env["GIT_CONFIG_COUNT"] = "1"
    env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {credentials}"

    subprocess.run(
        ["git", "clone", "https://github.com/RanaShoaaib/RoadGuard.git", str(REPO)],
        check=True,
        env=env,
    )

    del token, credentials, env

# Install RoadGuard
os.chdir(REPO)
subprocess.run(["python", "-m", "pip", "install", "-e", "."], check=True)

print("Repository:", REPO)

In [ ]:
import sys
sys.path.insert(0, str(REPO / "src"))

# Verify the import before starting training.
from roadguard.training.experiment import run_experiment
print("RoadGuard imported successfully")

## 2. Stage the prepared dataset locally

Notebook 1 saved one TAR package per archive in `RoadGuard_Processed/towns/` and the manifest and LOFO splits in `RoadGuard_Processed/processed/`. Unpacking them to the Colab disk avoids repeated Google Drive reads during training. Allow sufficient local disk space for the unpacked files.

In [ ]:
from roadguard.data.preparation import stage_prepared_dataset

DRIVE_PREPARED = Path("/content/drive/MyDrive/RoadGuard_Processed")
STAGE = Path("/content/roadguard_data")

staged = stage_prepared_dataset(DRIVE_PREPARED, STAGE)
DATA_ROOT = staged["data_root"]
LOFO_DIR = staged["processed_dir"] / "lofo_splits"
print("Dataset staged:", DATA_ROOT)
print("LOFO folds:", sorted(p.name for p in LOFO_DIR.glob("*.csv")))


## 3. Run the configured experiment

The YAML selects architecture, training length and the recipe (scenario-balanced sampling, focal+Dice loss, EMA weights, cosine schedule, mild RGB augmentation, 384×576 input scored at the original 512×768 resolution, smoothed validation-AP checkpoint selection). This cell trains the teacher, baseline and student for each held-out family in turn.

- Finished families are skipped, so rerunning after a disconnect continues where it stopped. A run cannot resume part-way: an unfinished family leaves a partial folder that you must delete on Drive before rerunning it.
- A failure in one family is printed and the loop continues with the next.
- When the loop ends, Drive is flushed and the GPU is released so the compute-unit meter stops (`RELEASE_GPU_WHEN_DONE`).

In [ ]:
import traceback
import torch
from roadguard.training.config import load_config
from roadguard.training.experiment import run_experiment

# EXPERIMENT CONFIGURATION (the YAML holds the recipe; override anything in build_config below)
ARCH = "unet"                                  # unet | segformer
FAMILIES = ["special", "home", "animal"]       # held-out (test) families, run in this order
SEED = 42
RELEASE_GPU_WHEN_DONE = True                   # free the GPU and stop the compute-unit meter at the end

OUTPUT_ROOT = Path("/content/drive/MyDrive/RoadGuard_Experiments")
CONFIG_PATH = REPO / "configs" / f"{ARCH}.yaml"
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before training")
print("GPU:", torch.cuda.get_device_name(0))


def build_config(family):
    config = load_config(CONFIG_PATH)
    config["experiment"].update(lofo_family=family, seed=SEED)
    # Optional overrides (the YAML holds the defaults):
    # config["data"]["image_size"] = [512, 768]      # native input, ~1.8x more compute
    # config["data"]["batch_size"] = 8               # if you run out of GPU memory
    # config["data"]["augment"]["probability"] = 0.0    # color changes + crop off (flip stays)
    return config


interrupted = False
try:
    for FAMILY in FAMILIES:
        config = build_config(FAMILY)
        run_dir = OUTPUT_ROOT / config["experiment"]["name"] / FAMILY / f"seed_{SEED}"
        if (run_dir / "report.json").exists():
            print(f"skip {FAMILY}: already finished"); continue
        if run_dir.exists() and any(run_dir.iterdir()):
            print(f"WARNING {FAMILY}: unfinished folder {run_dir}; delete it on Drive to rerun"); continue
        print(f"\n=== {ARCH} | held-out family: {FAMILY} | seed {SEED} ===")
        print("Experiment:", config["experiment"])
        try:
            run_experiment(config, LOFO_DIR / f"{FAMILY}.csv", DATA_ROOT, OUTPUT_ROOT, device="cuda")
        except Exception:
            print(f"FAILED {FAMILY}:"); traceback.print_exc()
except KeyboardInterrupt:
    interrupted = True            # a manual stop keeps the runtime so you can inspect it
    raise
finally:
    if RELEASE_GPU_WHEN_DONE and not interrupted:
        from google.colab import drive, runtime
        drive.flush_and_unmount()  # make sure the last files reached Drive
        runtime.unassign()         # releases the GPU, so the meter stops


## 4. Compare held-out test performance

The primary metric is **macro scenario AP**: average precision computed within each held-out scenario, then averaged, so large scenarios do not dominate. Dice, precision and recall use a threshold chosen on validation and applied once to test. *Selected* is the checkpoint picked by smoothed validation AP; *last* is the final epoch.

In [ ]:
import pandas as pd
from roadguard.evaluation.reports import load_reports

config = build_config(FAMILIES[0])
table = load_reports(OUTPUT_ROOT, [config["experiment"]["name"]], FAMILIES, [SEED],
                     roles=config["experiment"]["roles"])
display(table.drop(columns=["run", "seed"]).round(4))
print("Saved experiment results:", OUTPUT_ROOT / config["experiment"]["name"])
